# Advanced NLP Models: Transformers, Fine-tuning, and State-of-the-Art

This notebook implements advanced NLP models including transformer-based architectures, fine-tuning strategies, and state-of-the-art techniques for various NLP tasks.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
import pandas as pd
from typing import Dict, List, Optional, Tuple, Union, Any
from dataclasses import dataclass, field
import json
import warnings
from pathlib import Path
import time
from collections import defaultdict
from tqdm import tqdm

# Transformers and related libraries
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    AutoModelForQuestionAnswering,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    AutoModelForMaskedLM,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    DataCollatorForLanguageModeling,
    pipeline,
    BertConfig,
    GPT2Config,
    T5Config
)

from datasets import load_dataset, Dataset as HFDataset
from evaluate import load
import accelerate

# Sentence transformers for embeddings
try:
    from sentence_transformers import SentenceTransformer, util
    SENTENCE_TRANSFORMERS_AVAILABLE = True
except ImportError:
    print("sentence-transformers not installed. Run: pip install sentence-transformers")
    SENTENCE_TRANSFORMERS_AVAILABLE = False

# Additional NLP libraries
import spacy
import nltk
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
from sklearn.model_selection import train_test_split

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Sentiment Analysis and Emotion Detection

In [ ]:
@dataclass
class SentimentConfig:
    """Configuration for sentiment analysis."""
    model_name: str = 'distilbert-base-uncased'
    num_labels: int = 3  # positive, negative, neutral
    max_length: int = 512
    batch_size: int = 16
    learning_rate: float = 2e-5
    num_epochs: int = 3
    warmup_steps: int = 500
    weight_decay: float = 0.01
    use_mixed_precision: bool = True
    gradient_accumulation_steps: int = 1


class AdvancedSentimentAnalyzer:
    """Advanced sentiment analysis with multiple models and techniques."""
    
    def __init__(self, config: Optional[SentimentConfig] = None):
        self.config = config or SentimentConfig()
        self.tokenizer = None
        self.model = None
        self.emotion_model = None
        self.aspect_model = None
        
        # Initialize pre-trained models
        self._initialize_models()
    
    def _initialize_models(self):
        """Initialize various sentiment and emotion models."""
        
        # Basic sentiment model
        self.sentiment_pipeline = pipeline(
            'sentiment-analysis',
            model='distilbert-base-uncased-finetuned-sst-2-english',
            device=0 if torch.cuda.is_available() else -1
        )
        
        # Emotion detection
        self.emotion_pipeline = pipeline(
            'text-classification',
            model='j-hartmann/emotion-english-distilroberta-base',
            device=0 if torch.cuda.is_available() else -1
        )
        
        # Financial sentiment (domain-specific)
        self.financial_pipeline = pipeline(
            'sentiment-analysis',
            model='ProsusAI/finbert',
            device=0 if torch.cuda.is_available() else -1
        )
    
    def analyze_sentiment(self, texts: Union[str, List[str]], 
                        domain: str = 'general') -> List[Dict]:
        """Analyze sentiment with domain-specific models."""
        
        if isinstance(texts, str):
            texts = [texts]
        
        results = []
        
        for text in texts:
            result = {'text': text[:100] + '...' if len(text) > 100 else text}
            
            # General sentiment
            if domain in ['general', 'all']:
                sentiment = self.sentiment_pipeline(text[:512])[0]
                result['sentiment'] = {
                    'label': sentiment['label'],
                    'score': sentiment['score']
                }
            
            # Emotion detection
            if domain in ['emotion', 'all']:
                emotion = self.emotion_pipeline(text[:512])[0]
                result['emotion'] = {
                    'label': emotion['label'],
                    'score': emotion['score']
                }
            
            # Financial sentiment
            if domain in ['financial', 'all']:
                fin_sentiment = self.financial_pipeline(text[:512])[0]
                result['financial_sentiment'] = {
                    'label': fin_sentiment['label'],
                    'score': fin_sentiment['score']
                }
            
            results.append(result)
        
        return results
    
    def fine_tune_sentiment_model(self, train_texts: List[str], train_labels: List[int],
                                 val_texts: List[str], val_labels: List[int]):
        """Fine-tune a sentiment model on custom data."""
        
        # Load tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(self.config.model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.config.model_name,
            num_labels=self.config.num_labels
        ).to(device)
        
        # Prepare datasets
        train_dataset = self._prepare_dataset(train_texts, train_labels)
        val_dataset = self._prepare_dataset(val_texts, val_labels)
        
        # Training arguments
        training_args = TrainingArguments(
            output_dir='./sentiment_model',
            num_train_epochs=self.config.num_epochs,
            per_device_train_batch_size=self.config.batch_size,
            per_device_eval_batch_size=self.config.batch_size,
            warmup_steps=self.config.warmup_steps,
            weight_decay=self.config.weight_decay,
            logging_dir='./logs',
            logging_steps=10,
            evaluation_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='f1',
            fp16=self.config.use_mixed_precision and torch.cuda.is_available(),
            gradient_accumulation_steps=self.config.gradient_accumulation_steps,
            dataloader_num_workers=4
        )
        
        # Create trainer
        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=self.tokenizer,
            compute_metrics=self._compute_metrics
        )
        
        # Train
        trainer.train()
        
        # Save model
        trainer.save_model('./sentiment_model_final')
        
        return trainer
    
    def _prepare_dataset(self, texts: List[str], labels: List[int]) -> HFDataset:
        """Prepare dataset for training."""
        
        # Tokenize texts
        encodings = self.tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=self.config.max_length,
            return_tensors='pt'
        )
        
        # Create dataset
        dataset = HFDataset.from_dict({
            'input_ids': encodings['input_ids'],
            'attention_mask': encodings['attention_mask'],
            'labels': labels
        })
        
        return dataset
    
    def _compute_metrics(self, eval_pred):
        """Compute metrics for evaluation."""
        
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        
        precision, recall, f1, _ = precision_recall_fscore_support(
            labels, predictions, average='weighted'
        )
        accuracy = accuracy_score(labels, predictions)
        
        return {
            'accuracy': accuracy,
            'f1': f1,
            'precision': precision,
            'recall': recall
        }
    
    def aspect_based_sentiment(self, text: str, aspects: List[str]) -> Dict:
        """Perform aspect-based sentiment analysis."""
        
        results = {}
        
        for aspect in aspects:
            # Create prompt for aspect sentiment
            prompt = f"What is the sentiment towards {aspect} in: {text}"
            
            # Use general sentiment as proxy (in practice, use specialized model)
            sentiment = self.sentiment_pipeline(prompt)[0]
            
            results[aspect] = {
                'sentiment': sentiment['label'],
                'confidence': sentiment['score']
            }
        
        return results

## 2. Named Entity Recognition and Relation Extraction

In [ ]:
class AdvancedNER:
    """Advanced Named Entity Recognition with multiple models."""
    
    def __init__(self):
        # Initialize multiple NER models
        self.models = {}
        self._initialize_models()
    
    def _initialize_models(self):
        """Initialize different NER models."""
        
        # Standard BERT NER
        self.models['bert'] = pipeline(
            'ner',
            model='dbmdz/bert-large-cased-finetuned-conll03-english',
            aggregation_strategy='simple',
            device=0 if torch.cuda.is_available() else -1
        )
        
        # RoBERTa NER
        self.models['roberta'] = pipeline(
            'ner',
            model='Jean-Baptiste/roberta-large-ner-english',
            aggregation_strategy='simple',
            device=0 if torch.cuda.is_available() else -1
        )
        
        # SpaCy NER
        self.spacy_nlp = spacy.load('en_core_web_sm')
    
    def extract_entities(self, text: str, model: str = 'ensemble') -> Dict:
        """Extract entities using specified model or ensemble."""
        
        if model == 'ensemble':
            return self._ensemble_ner(text)
        elif model in self.models:
            return self._single_model_ner(text, model)
        else:
            raise ValueError(f"Unknown model: {model}")
    
    def _single_model_ner(self, text: str, model_name: str) -> Dict:
        """Extract entities using a single model."""
        
        if model_name == 'spacy':
            doc = self.spacy_nlp(text)
            entities = []
            for ent in doc.ents:
                entities.append({
                    'entity': ent.text,
                    'label': ent.label_,
                    'start': ent.start_char,
                    'end': ent.end_char
                })
        else:
            entities = self.models[model_name](text)
        
        return {
            'model': model_name,
            'entities': entities
        }
    
    def _ensemble_ner(self, text: str) -> Dict:
        """Ensemble NER using multiple models."""
        
        all_entities = defaultdict(list)
        
        # Get entities from each model
        for model_name in ['bert', 'roberta']:
            try:
                entities = self.models[model_name](text)
                for ent in entities:
                    key = (ent['word'], ent['entity_group'])
                    all_entities[key].append({
                        'model': model_name,
                        'score': ent['score'],
                        'start': ent['start'],
                        'end': ent['end']
                    })
            except:
                pass
        
        # SpaCy entities
        doc = self.spacy_nlp(text)
        for ent in doc.ents:
            key = (ent.text, ent.label_)
            all_entities[key].append({
                'model': 'spacy',
                'score': 1.0,
                'start': ent.start_char,
                'end': ent.end_char
            })
        
        # Aggregate results
        final_entities = []
        for (text, label), sources in all_entities.items():
            avg_score = np.mean([s['score'] for s in sources])
            final_entities.append({
                'entity': text,
                'label': label,
                'confidence': avg_score,
                'num_models': len(sources),
                'models': [s['model'] for s in sources]
            })
        
        # Sort by confidence
        final_entities.sort(key=lambda x: x['confidence'], reverse=True)
        
        return {
            'model': 'ensemble',
            'entities': final_entities
        }
    
    def extract_relations(self, text: str) -> List[Dict]:
        """Extract relations between entities."""
        
        # Get entities first
        entities = self.extract_entities(text)['entities']
        
        # Simple dependency-based relation extraction
        doc = self.spacy_nlp(text)
        relations = []
        
        for sent in doc.sents:
            # Find root verb
            root = [token for token in sent if token.dep_ == 'ROOT'][0] if [token for token in sent if token.dep_ == 'ROOT'] else None
            
            if root and root.pos_ == 'VERB':
                subjects = [token for token in root.children if 'subj' in token.dep_]
                objects = [token for token in root.children if 'obj' in token.dep_]
                
                for subj in subjects:
                    for obj in objects:
                        # Check if they are entities
                        subj_entity = self._find_entity(subj.text, entities)
                        obj_entity = self._find_entity(obj.text, entities)
                        
                        if subj_entity and obj_entity:
                            relations.append({
                                'subject': subj_entity['entity'],
                                'relation': root.lemma_,
                                'object': obj_entity['entity'],
                                'confidence': (subj_entity.get('confidence', 1.0) + 
                                             obj_entity.get('confidence', 1.0)) / 2
                            })
        
        return relations
    
    def _find_entity(self, text: str, entities: List[Dict]) -> Optional[Dict]:
        """Find entity in list."""
        for entity in entities:
            if text in entity.get('entity', entity.get('word', '')):
                return entity
        return None
    
    def train_custom_ner(self, train_data: List[Tuple[str, List[Tuple]]], 
                        model_name: str = 'bert-base-cased'):
        """Train custom NER model."""
        
        # Load tokenizer and model
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForTokenClassification.from_pretrained(
            model_name,
            num_labels=9  # B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC, B-MISC, I-MISC, O
        ).to(device)
        
        # Prepare data
        train_dataset = self._prepare_ner_dataset(train_data, tokenizer)
        
        # Training arguments
        training_args = TrainingArguments(
            output_dir='./ner_model',
            num_train_epochs=3,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            warmup_steps=500,
            weight_decay=0.01,
            logging_dir='./logs'
        )
        
        # Create trainer
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            tokenizer=tokenizer
        )
        
        # Train
        trainer.train()
        
        return trainer
    
    def _prepare_ner_dataset(self, data: List[Tuple[str, List[Tuple]]], tokenizer) -> HFDataset:
        """Prepare NER dataset."""
        # Implementation would prepare the dataset for NER training
        pass

## 3. Question Answering System

In [ ]:
class AdvancedQuestionAnswering:
    """Advanced question answering system with multiple approaches."""
    
    def __init__(self):
        self.models = {}
        self._initialize_models()
        
        # Initialize retriever for RAG
        if SENTENCE_TRANSFORMERS_AVAILABLE:
            self.encoder = SentenceTransformer('all-MiniLM-L6-v2')
            self.document_embeddings = None
            self.documents = None
    
    def _initialize_models(self):
        """Initialize QA models."""
        
        # Extractive QA
        self.models['extractive'] = pipeline(
            'question-answering',
            model='deepset/roberta-base-squad2',
            device=0 if torch.cuda.is_available() else -1
        )
        
        # Generative QA
        self.models['generative'] = pipeline(
            'text2text-generation',
            model='google/flan-t5-base',
            device=0 if torch.cuda.is_available() else -1
        )
    
    def answer_question(self, question: str, context: str = None, 
                       method: str = 'extractive') -> Dict:
        """Answer question using specified method."""
        
        if method == 'extractive':
            return self._extractive_qa(question, context)
        elif method == 'generative':
            return self._generative_qa(question, context)
        elif method == 'rag':
            return self._rag_qa(question)
        else:
            raise ValueError(f"Unknown method: {method}")
    
    def _extractive_qa(self, question: str, context: str) -> Dict:
        """Extractive question answering."""
        
        if not context:
            return {'answer': 'No context provided', 'score': 0.0}
        
        result = self.models['extractive'](
            question=question,
            context=context
        )
        
        return {
            'answer': result['answer'],
            'score': result['score'],
            'start': result['start'],
            'end': result['end'],
            'method': 'extractive'
        }
    
    def _generative_qa(self, question: str, context: str = None) -> Dict:
        """Generative question answering."""
        
        if context:
            prompt = f"Context: {context}\n\nQuestion: {question}\n\nAnswer:"
        else:
            prompt = f"Question: {question}\n\nAnswer:"
        
        result = self.models['generative'](
            prompt,
            max_length=150,
            temperature=0.7
        )
        
        return {
            'answer': result[0]['generated_text'],
            'method': 'generative'
        }
    
    def _rag_qa(self, question: str) -> Dict:
        """Retrieval-augmented generation QA."""
        
        if not SENTENCE_TRANSFORMERS_AVAILABLE or self.documents is None:
            return {'answer': 'RAG not available or no documents indexed', 'score': 0.0}
        
        # Encode question
        question_embedding = self.encoder.encode(question, convert_to_tensor=True)
        
        # Find most relevant documents
        similarities = util.pytorch_cos_sim(question_embedding, self.document_embeddings)[0]
        top_k = min(3, len(self.documents))
        top_indices = torch.topk(similarities, k=top_k).indices
        
        # Combine top documents as context
        context = ' '.join([self.documents[idx] for idx in top_indices])
        
        # Use extractive QA on retrieved context
        return self._extractive_qa(question, context)
    
    def index_documents(self, documents: List[str]):
        """Index documents for RAG."""
        
        if not SENTENCE_TRANSFORMERS_AVAILABLE:
            print("Sentence transformers not available")
            return
        
        self.documents = documents
        self.document_embeddings = self.encoder.encode(
            documents,
            convert_to_tensor=True,
            show_progress_bar=True
        )
        
        print(f"Indexed {len(documents)} documents")
    
    def create_qa_dataset(self, texts: List[str], 
                         questions: List[str],
                         answers: List[str]) -> HFDataset:
        """Create QA dataset for training."""
        
        dataset = HFDataset.from_dict({
            'context': texts,
            'question': questions,
            'answers': [{'text': [ans], 'answer_start': [0]} for ans in answers]
        })
        
        return dataset

## 4. Text Generation and Language Modeling

In [ ]:
class AdvancedTextGeneration:
    """Advanced text generation with various models and techniques."""
    
    def __init__(self):
        self.models = {}
        self._initialize_models()
    
    def _initialize_models(self):
        """Initialize text generation models."""
        
        # GPT-2 for general text generation
        self.models['gpt2'] = pipeline(
            'text-generation',
            model='gpt2-medium',
            device=0 if torch.cuda.is_available() else -1
        )
        
        # T5 for conditional generation
        self.models['t5'] = pipeline(
            'text2text-generation',
            model='t5-base',
            device=0 if torch.cuda.is_available() else -1
        )
        
        # BART for summarization and generation
        self.models['bart'] = pipeline(
            'summarization',
            model='facebook/bart-large-cnn',
            device=0 if torch.cuda.is_available() else -1
        )
    
    def generate_text(self, prompt: str, 
                     model: str = 'gpt2',
                     max_length: int = 200,
                     temperature: float = 0.8,
                     top_p: float = 0.9,
                     num_return_sequences: int = 1) -> List[str]:
        """Generate text using specified model."""
        
        if model not in self.models:
            raise ValueError(f"Unknown model: {model}")
        
        if model == 'gpt2':
            results = self.models['gpt2'](
                prompt,
                max_length=max_length,
                temperature=temperature,
                top_p=top_p,
                num_return_sequences=num_return_sequences,
                pad_token_id=50256  # GPT2 pad token
            )
            return [r['generated_text'] for r in results]
        
        elif model == 't5':
            results = self.models['t5'](
                prompt,
                max_length=max_length,
                temperature=temperature,
                num_return_sequences=num_return_sequences
            )
            return [r['generated_text'] for r in results]
        
        else:
            return ["Model not configured for general text generation"]
    
    def paraphrase(self, text: str, num_paraphrases: int = 3) -> List[str]:
        """Generate paraphrases of input text."""
        
        prompt = f"paraphrase: {text}"
        
        paraphrases = self.models['t5'](
            prompt,
            max_length=len(text.split()) * 2,
            num_return_sequences=num_paraphrases,
            temperature=0.9
        )
        
        return [p['generated_text'] for p in paraphrases]
    
    def complete_code(self, code_snippet: str, language: str = 'python') -> str:
        """Complete code snippet."""
        
        prompt = f"Complete the following {language} code:\n{code_snippet}\n"
        
        completion = self.models['gpt2'](
            prompt,
            max_length=len(code_snippet) + 100,
            temperature=0.2,
            top_p=0.95
        )
        
        return completion[0]['generated_text']
    
    def generate_summary(self, text: str, 
                        max_length: int = 150,
                        min_length: int = 50) -> str:
        """Generate summary of text."""
        
        summary = self.models['bart'](
            text,
            max_length=max_length,
            min_length=min_length,
            do_sample=False
        )
        
        return summary[0]['summary_text']
    
    def style_transfer(self, text: str, target_style: str) -> str:
        """Transfer text to target style."""
        
        style_prompts = {
            'formal': 'Rewrite the following text in a formal style: ',
            'casual': 'Rewrite the following text in a casual style: ',
            'technical': 'Rewrite the following text using technical language: ',
            'simple': 'Simplify the following text: ',
            'poetic': 'Rewrite the following text poetically: '
        }
        
        if target_style not in style_prompts:
            return text
        
        prompt = style_prompts[target_style] + text
        
        result = self.models['t5'](
            prompt,
            max_length=len(text.split()) * 2,
            temperature=0.8
        )
        
        return result[0]['generated_text']
    
    def fine_tune_generator(self, train_texts: List[str], 
                          model_name: str = 'gpt2'):
        """Fine-tune text generation model."""
        
        # Load tokenizer and model
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
        
        # Add padding token
        tokenizer.pad_token = tokenizer.eos_token
        
        # Tokenize texts
        encodings = tokenizer(
            train_texts,
            truncation=True,
            padding=True,
            max_length=512,
            return_tensors='pt'
        )
        
        # Create dataset
        dataset = HFDataset.from_dict({
            'input_ids': encodings['input_ids'],
            'attention_mask': encodings['attention_mask'],
            'labels': encodings['input_ids']  # For language modeling
        })
        
        # Data collator
        data_collator = DataCollatorForLanguageModeling(
            tokenizer=tokenizer,
            mlm=False  # Causal LM, not masked LM
        )
        
        # Training arguments
        training_args = TrainingArguments(
            output_dir='./generator_model',
            num_train_epochs=3,
            per_device_train_batch_size=4,
            warmup_steps=500,
            weight_decay=0.01,
            logging_dir='./logs',
            save_strategy='epoch'
        )
        
        # Create trainer
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=dataset,
            data_collator=data_collator,
            tokenizer=tokenizer
        )
        
        # Train
        trainer.train()
        
        return trainer

## 5. Semantic Search and Similarity

In [ ]:
class SemanticSearch:
    """Semantic search and similarity using sentence embeddings."""
    
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        if not SENTENCE_TRANSFORMERS_AVAILABLE:
            raise ImportError("sentence-transformers required")
        
        self.encoder = SentenceTransformer(model_name)
        self.index = None
        self.documents = None
        self.embeddings = None
    
    def index_documents(self, documents: List[str], 
                       batch_size: int = 32):
        """Index documents for semantic search."""
        
        print(f"Indexing {len(documents)} documents...")
        
        self.documents = documents
        self.embeddings = self.encoder.encode(
            documents,
            batch_size=batch_size,
            convert_to_tensor=True,
            show_progress_bar=True
        )
        
        # Optionally use FAISS for large-scale search
        try:
            import faiss
            
            # Create FAISS index
            dimension = self.embeddings.shape[1]
            self.index = faiss.IndexFlatL2(dimension)
            self.index.add(self.embeddings.cpu().numpy())
            print("FAISS index created")
        except ImportError:
            print("FAISS not available, using cosine similarity")
    
    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        """Search for similar documents."""
        
        if self.documents is None:
            return []
        
        # Encode query
        query_embedding = self.encoder.encode(
            query,
            convert_to_tensor=True
        )
        
        if self.index is not None:
            # Use FAISS
            distances, indices = self.index.search(
                query_embedding.cpu().numpy().reshape(1, -1),
                top_k
            )
            
            results = []
            for idx, dist in zip(indices[0], distances[0]):
                results.append({
                    'document': self.documents[idx],
                    'score': float(1 / (1 + dist)),  # Convert distance to similarity
                    'index': int(idx)
                })
        else:
            # Use cosine similarity
            similarities = util.pytorch_cos_sim(
                query_embedding,
                self.embeddings
            )[0]
            
            top_results = torch.topk(similarities, k=min(top_k, len(self.documents)))
            
            results = []
            for score, idx in zip(top_results.values, top_results.indices):
                results.append({
                    'document': self.documents[idx],
                    'score': float(score),
                    'index': int(idx)
                })
        
        return results
    
    def compute_similarity(self, text1: str, text2: str) -> float:
        """Compute similarity between two texts."""
        
        embeddings = self.encoder.encode([text1, text2], convert_to_tensor=True)
        similarity = util.pytorch_cos_sim(embeddings[0], embeddings[1])
        
        return float(similarity)
    
    def find_duplicates(self, threshold: float = 0.9) -> List[Tuple]:
        """Find duplicate or near-duplicate documents."""
        
        if self.embeddings is None:
            return []
        
        # Compute pairwise similarities
        similarities = util.pytorch_cos_sim(self.embeddings, self.embeddings)
        
        duplicates = []
        n = len(self.documents)
        
        for i in range(n):
            for j in range(i + 1, n):
                if similarities[i][j] > threshold:
                    duplicates.append((i, j, float(similarities[i][j])))
        
        return duplicates
    
    def cluster_documents(self, n_clusters: int = 5) -> Dict:
        """Cluster documents based on semantic similarity."""
        
        if self.embeddings is None:
            return {}
        
        from sklearn.cluster import KMeans
        
        # Perform clustering
        clustering = KMeans(n_clusters=n_clusters, random_state=42)
        cluster_labels = clustering.fit_predict(self.embeddings.cpu().numpy())
        
        # Organize results
        clusters = defaultdict(list)
        for idx, label in enumerate(cluster_labels):
            clusters[int(label)].append({
                'document': self.documents[idx],
                'index': idx
            })
        
        return dict(clusters)

## 6. Example Usage and Demonstrations

In [ ]:
def demo_sentiment_analysis():
    """Demonstrate sentiment analysis capabilities."""
    
    print("\n" + "=" * 50)
    print("Sentiment Analysis Demo")
    print("=" * 50)
    
    analyzer = AdvancedSentimentAnalyzer()
    
    texts = [
        "I absolutely love this product! It exceeded all my expectations.",
        "The service was terrible and the food was cold.",
        "It's okay, nothing special but not bad either.",
        "The stock market showed strong gains today, with tech stocks leading the rally."
    ]
    
    # Analyze sentiment
    results = analyzer.analyze_sentiment(texts, domain='all')
    
    for result in results:
        print(f"\nText: {result['text']}")
        if 'sentiment' in result:
            print(f"Sentiment: {result['sentiment']['label']} ({result['sentiment']['score']:.3f})")
        if 'emotion' in result:
            print(f"Emotion: {result['emotion']['label']} ({result['emotion']['score']:.3f})")
        if 'financial_sentiment' in result:
            print(f"Financial: {result['financial_sentiment']['label']} ({result['financial_sentiment']['score']:.3f})")
    
    return results

def demo_ner():
    """Demonstrate NER capabilities."""
    
    print("\n" + "=" * 50)
    print("Named Entity Recognition Demo")
    print("=" * 50)
    
    ner = AdvancedNER()
    
    text = """
    Apple Inc. CEO Tim Cook announced a partnership with Samsung in Seoul, South Korea.
    The deal, worth $5 billion, will begin on January 1, 2024.
    """
    
    # Extract entities
    results = ner.extract_entities(text, model='ensemble')
    
    print("\nExtracted Entities:")
    for entity in results['entities'][:10]:
        print(f"  - {entity['entity']} ({entity['label']}) - Confidence: {entity['confidence']:.3f}")
    
    # Extract relations
    relations = ner.extract_relations(text)
    
    if relations:
        print("\nExtracted Relations:")
        for rel in relations:
            print(f"  - {rel['subject']} --{rel['relation']}--> {rel['object']}")
    
    return results, relations

def demo_qa():
    """Demonstrate question answering."""
    
    print("\n" + "=" * 50)
    print("Question Answering Demo")
    print("=" * 50)
    
    qa = AdvancedQuestionAnswering()
    
    context = """
    The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.
    It is named after the engineer Gustave Eiffel, whose company designed and built the tower.
    Construction began in 1887 and was completed in 1889. The tower is 330 meters tall.
    """
    
    questions = [
        "Who designed the Eiffel Tower?",
        "When was the Eiffel Tower built?",
        "How tall is the Eiffel Tower?"
    ]
    
    for question in questions:
        # Extractive QA
        answer_ext = qa.answer_question(question, context, method='extractive')
        print(f"\nQ: {question}")
        print(f"A (Extractive): {answer_ext['answer']} (Score: {answer_ext.get('score', 0):.3f})")
        
        # Generative QA
        answer_gen = qa.answer_question(question, context, method='generative')
        print(f"A (Generative): {answer_gen['answer']}")
    
    return True

def demo_text_generation():
    """Demonstrate text generation."""
    
    print("\n" + "=" * 50)
    print("Text Generation Demo")
    print("=" * 50)
    
    generator = AdvancedTextGeneration()
    
    # Text generation
    prompt = "The future of artificial intelligence will"
    generated = generator.generate_text(prompt, model='gpt2', max_length=100)
    print(f"\nPrompt: {prompt}")
    print(f"Generated: {generated[0]}")
    
    # Paraphrasing
    text = "Machine learning is a subset of artificial intelligence that enables computers to learn from data."
    paraphrases = generator.paraphrase(text, num_paraphrases=2)
    print(f"\nOriginal: {text}")
    for i, para in enumerate(paraphrases, 1):
        print(f"Paraphrase {i}: {para}")
    
    # Style transfer
    casual_text = "Hey, wanna grab some food later?"
    formal = generator.style_transfer(casual_text, 'formal')
    print(f"\nCasual: {casual_text}")
    print(f"Formal: {formal}")
    
    return True

def demo_semantic_search():
    """Demonstrate semantic search."""
    
    if not SENTENCE_TRANSFORMERS_AVAILABLE:
        print("Sentence transformers not available")
        return
    
    print("\n" + "=" * 50)
    print("Semantic Search Demo")
    print("=" * 50)
    
    search_engine = SemanticSearch()
    
    # Sample documents
    documents = [
        "Python is a high-level programming language.",
        "Machine learning enables computers to learn from data.",
        "Deep learning is a subset of machine learning using neural networks.",
        "Natural language processing helps computers understand human language.",
        "Computer vision allows machines to interpret visual information.",
        "Data science combines statistics, programming, and domain knowledge."
    ]
    
    # Index documents
    search_engine.index_documents(documents)
    
    # Search
    query = "What is artificial intelligence?"
    results = search_engine.search(query, top_k=3)
    
    print(f"\nQuery: {query}")
    print("\nTop Results:")
    for i, result in enumerate(results, 1):
        print(f"{i}. {result['document']} (Score: {result['score']:.3f})")
    
    # Find duplicates
    duplicates = search_engine.find_duplicates(threshold=0.8)
    if duplicates:
        print("\nPotential Duplicates:")
        for i, j, score in duplicates:
            print(f"  - Doc {i} & Doc {j}: {score:.3f}")
    
    return results

# Run all demos
if __name__ == "__main__":
    print("Running Advanced NLP Models Demonstrations...\n")
    
    # Run demonstrations
    sentiment_results = demo_sentiment_analysis()
    ner_results, relations = demo_ner()
    qa_results = demo_qa()
    gen_results = demo_text_generation()
    search_results = demo_semantic_search()
    
    print("\n" + "=" * 50)
    print("All demonstrations completed successfully!")
    print("=" * 50)

## Summary

This advanced NLP models notebook provides:

1. **Sentiment Analysis**: Multi-domain sentiment and emotion detection with fine-tuning capabilities
2. **Named Entity Recognition**: Ensemble NER with relation extraction
3. **Question Answering**: Extractive, generative, and RAG-based QA systems
4. **Text Generation**: Various generation models with style transfer and paraphrasing
5. **Semantic Search**: Document indexing, similarity search, and clustering
6. **Fine-tuning**: Custom model training for specific tasks
7. **Production Ready**: Optimized for both CPU and GPU deployment

All models are state-of-the-art transformers with support for custom training and domain adaptation.